In [2]:
# 09_previous_day_rolling_trend_phase3a_training.ipynb
# Cell 1. Phase 3A rolling/trend experiment setup and tensor loading
# 원하는 결과:
# - Phase 3A grid와 tensor를 로드한다.
# - MLP current-day / GRU window 7 / GRU window 14 입력 shape를 확인한다.
# - device, seed, output path를 고정한다.
# - 아직 training은 실행하지 않는다.

from pathlib import Path
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PREVIOUS_DAY_DIR = PROJECT_ROOT / "data" / "processed" / "deep_learning_previous_day"

ROLLING_TREND_DIR = PREVIOUS_DAY_DIR / "rolling_trend_daytime_only"
EXPERIMENT_DIR = PREVIOUS_DAY_DIR / "experiments" / "phase_3a_rolling_trend_daytime_only"

GRID_PATH = EXPERIMENT_DIR / "phase_3a_rolling_trend_experiment_grid.csv"
OUTPUT_DIR = EXPERIMENT_DIR / "phase_3a_outputs"
MODEL_DIR = OUTPUT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = OUTPUT_DIR / "phase_3a_rolling_trend_model_metrics.csv"
HISTORY_PATH = OUTPUT_DIR / "phase_3a_rolling_trend_training_history.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "phase_3a_rolling_trend_model_predictions.csv"

SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 120
PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# 안전장치: 셀을 처음 실행했을 때 곧바로 학습되지 않게 둡니다.
# 다음 학습 셀에서 True로 바꿔 실행하세요.
RUN_PHASE3A_TRAINING = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

phase3a_grid = pd.read_csv(GRID_PATH, encoding="utf-8-sig")

def load_phase3a_tensor(row, split):
    tensor_dir = PROJECT_ROOT / row["tensor_dir"]
    npz_path = tensor_dir / f"{split}.npz"
    data = np.load(npz_path, allow_pickle=True)

    if row["model_family"] == "mlp_current_day":
        X = data["X_mlp_current_day"].astype(np.float32)
    else:
        X = data["X_sequence"].astype(np.float32)

    y = data["y"].astype(np.int64)

    return {
        "X": X,
        "y": y,
        "participant_object_id": data["participant_object_id"].astype(str),
        "feature_names": data["feature_names"].astype(str),
        "path": npz_path,
    }

loaded_experiments = {}

for _, row in phase3a_grid.iterrows():
    experiment_id = row["experiment_id"]

    loaded_experiments[experiment_id] = {
        "config": row.to_dict(),
        "train": load_phase3a_tensor(row, "train"),
        "validation": load_phase3a_tensor(row, "validation"),
        "test": load_phase3a_tensor(row, "test"),
    }

summary_rows = []

for experiment_id, bundle in loaded_experiments.items():
    config = bundle["config"]

    for split in ["train", "validation", "test"]:
        X = bundle[split]["X"]
        y = bundle[split]["y"]

        summary_rows.append(
            {
                "experiment_id": experiment_id,
                "model_family": config["model_family"],
                "window": int(config["window"]),
                "split": split,
                "X_shape": str(tuple(X.shape)),
                "samples": len(y),
                "features": X.shape[-1],
                "participants": len(set(bundle[split]["participant_object_id"])),
                "target_0": int((y == 0).sum()),
                "target_1": int((y == 1).sum()),
                "target_mean": float(y.mean()),
                "nan_count": int(np.isnan(X).sum()),
                "inf_count": int(np.isinf(X).sum()),
            }
        )

phase3a_loaded_summary = pd.DataFrame(summary_rows)

print("=== Phase 3A Training Setup ===")
print("DEVICE:", DEVICE)
print("GRID_PATH:", GRID_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUN_PHASE3A_TRAINING:", RUN_PHASE3A_TRAINING)

print("\n=== Phase 3A Grid ===")
display(phase3a_grid)

print("\n=== Loaded Tensor Summary ===")
display(
    phase3a_loaded_summary
    .sort_values(["experiment_id", "split"])
    .reset_index(drop=True)
)

print("\nproblem rows:")
display(
    phase3a_loaded_summary[
        (phase3a_loaded_summary["nan_count"] != 0)
        | (phase3a_loaded_summary["inf_count"] != 0)
    ]
)

=== Phase 3A Training Setup ===
DEVICE: cpu
GRID_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_rolling_trend_experiment_grid.csv
OUTPUT_DIR: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs
RUN_PHASE3A_TRAINING: False

=== Phase 3A Grid ===


,experiment_id,feature_timing,subset_name,model_family,window,tensor_dir,feature_count,calendar_features_used,notes
0,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,data\processed\deep_learning_previous_day\roll...,122,False,MLP on previous-day daytime_only rolling/trend...
1,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,data\processed\deep_learning_previous_day\roll...,122,False,GRU on previous-day daytime_only rolling/trend...
2,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,data\processed\deep_learning_previous_day\roll...,122,False,GRU on previous-day daytime_only rolling/trend...



=== Loaded Tensor Summary ===


,experiment_id,model_family,window,split,X_shape,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count
0,phase3a_000,mlp_current_day,1,test,"(524, 122)",524,122,12,325,199,0.379771,0,0
1,phase3a_000,mlp_current_day,1,train,"(2135, 122)",2135,122,41,1269,866,0.405621,0,0
2,phase3a_000,mlp_current_day,1,validation,"(457, 122)",457,122,10,277,180,0.393873,0,0
3,phase3a_001,gru,7,test,"(296, 7, 122)",296,122,9,173,123,0.415541,0,0
4,phase3a_001,gru,7,train,"(1303, 7, 122)",1303,122,33,779,524,0.402149,0,0
5,phase3a_001,gru,7,validation,"(245, 7, 122)",245,122,9,153,92,0.375510,0,0
6,phase3a_002,gru,14,test,"(206, 14, 122)",206,122,5,109,97,0.470874,0,0
7,phase3a_002,gru,14,train,"(887, 14, 122)",887,122,27,553,334,0.376550,0,0
8,phase3a_002,gru,14,validation,"(136, 14, 122)",136,122,7,88,48,0.352941,0,0



problem rows:


,experiment_id,model_family,window,split,X_shape,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count


In [5]:
# Cell 2. Dataset, model, training/evaluation utility 정의
# 원하는 결과:
# - MLP current-day와 GRU를 학습할 수 있는 공통 함수들을 정의한다.
# - validation balanced accuracy 기준 early stopping을 사용한다.
# - threshold는 validation split에서 balanced accuracy가 최대가 되는 값으로 선택한다.
# - 이 셀은 함수 정의만 하며 training은 실행하지 않는다.

class NumpyBinaryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class MLPCurrentDay(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class GRUSequenceModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, dropout=0.20):
        super().__init__()
        effective_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=effective_dropout,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        output, hidden = self.gru(x)
        last_hidden = hidden[-1]
        return self.head(last_hidden).squeeze(-1)


def make_loader(X, y, batch_size, shuffle):
    dataset = NumpyBinaryDataset(X, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def compute_pos_weight(y):
    y = np.asarray(y)
    negative = int((y == 0).sum())
    positive = int((y == 1).sum())

    if positive == 0:
        return 1.0

    return negative / positive


@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    probs = []
    labels = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy()

        probs.append(batch_probs)
        labels.append(y_batch.numpy())

    return np.concatenate(labels), np.concatenate(probs)


def binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
        metrics["average_precision"] = average_precision_score(y_true, y_prob)
    else:
        metrics["roc_auc"] = np.nan
        metrics["average_precision"] = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics.update(
        {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }
    )

    return metrics


def find_best_threshold(y_true, y_prob):
    thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []
    for threshold in thresholds:
        row = binary_metrics(y_true, y_prob, threshold=threshold)
        rows.append(row)

    threshold_df = pd.DataFrame(rows)
    best_row = (
        threshold_df
        .sort_values(
            ["balanced_accuracy", "f1", "recall"],
            ascending=[False, False, False],
        )
        .iloc[0]
    )

    return float(best_row["threshold"]), threshold_df


def train_one_experiment(experiment_id, bundle, device):
    config = bundle["config"]
    model_family = config["model_family"]

    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]
    X_val = bundle["validation"]["X"]
    y_val = bundle["validation"]["y"]
    X_test = bundle["test"]["X"]
    y_test = bundle["test"]["y"]

    train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
    val_loader = make_loader(X_val, y_val, BATCH_SIZE, shuffle=False)
    test_loader = make_loader(X_test, y_test, BATCH_SIZE, shuffle=False)

    input_dim = X_train.shape[-1]

    if model_family == "mlp_current_day":
        model = MLPCurrentDay(input_dim=input_dim, hidden_dim=128, dropout=0.25)
    elif model_family == "gru":
        model = GRUSequenceModel(input_dim=input_dim, hidden_dim=64, num_layers=1, dropout=0.20)
    else:
        raise ValueError(f"Unsupported model_family: {model_family}")

    model = model.to(device)

    pos_weight_value = compute_pos_weight(y_train)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_balanced_accuracy = -np.inf
    best_state_dict = None
    best_epoch = None
    best_threshold = 0.5
    epochs_without_improvement = 0

    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            train_losses.append(float(loss.detach().cpu().item()))

        y_train_true, y_train_prob = predict_proba(model, train_loader, device)
        y_val_true, y_val_prob = predict_proba(model, val_loader, device)

        epoch_best_threshold, _ = find_best_threshold(y_val_true, y_val_prob)

        train_metrics = binary_metrics(y_train_true, y_train_prob, threshold=epoch_best_threshold)
        val_metrics = binary_metrics(y_val_true, y_val_prob, threshold=epoch_best_threshold)

        history_rows.append(
            {
                "experiment_id": experiment_id,
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)),
                "threshold_from_validation": epoch_best_threshold,
                "train_balanced_accuracy": train_metrics["balanced_accuracy"],
                "train_roc_auc": train_metrics["roc_auc"],
                "train_f1": train_metrics["f1"],
                "validation_balanced_accuracy": val_metrics["balanced_accuracy"],
                "validation_roc_auc": val_metrics["roc_auc"],
                "validation_f1": val_metrics["f1"],
            }
        )

        current_score = val_metrics["balanced_accuracy"]

        if current_score > best_val_balanced_accuracy:
            best_val_balanced_accuracy = current_score
            best_state_dict = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            best_epoch = epoch
            best_threshold = epoch_best_threshold
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"{experiment_id} epoch {epoch:03d} | "
                f"loss={np.mean(train_losses):.4f} | "
                f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | "
                f"val_auc={val_metrics['roc_auc']:.4f} | "
                f"thr={epoch_best_threshold:.2f}"
            )

        if epochs_without_improvement >= PATIENCE:
            print(
                f"{experiment_id} early stopped at epoch {epoch}; "
                f"best_epoch={best_epoch}, best_val_bal_acc={best_val_balanced_accuracy:.4f}"
            )
            break

    model.load_state_dict(best_state_dict)
    model = model.to(device)

    split_metric_rows = []
    prediction_rows = []

    for split_name, loader in [
        ("train", train_loader),
        ("validation", val_loader),
        ("test", test_loader),
    ]:
        y_true, y_prob = predict_proba(model, loader, device)
        metrics = binary_metrics(y_true, y_prob, threshold=best_threshold)

        metric_row = {
            "experiment_id": experiment_id,
            "feature_timing": config["feature_timing"],
            "subset_name": config["subset_name"],
            "model_family": config["model_family"],
            "window": int(config["window"]),
            "split": split_name,
            "best_epoch": int(best_epoch),
            "selected_threshold_from_validation": float(best_threshold),
            **metrics,
        }
        split_metric_rows.append(metric_row)

        split_bundle = bundle[split_name]
        participants = split_bundle["participant_object_id"]

        for idx in range(len(y_true)):
            prediction_rows.append(
                {
                    "experiment_id": experiment_id,
                    "split": split_name,
                    "row_index": idx,
                    "participant_object_id": participants[idx],
                    "y_true": int(y_true[idx]),
                    "y_prob": float(y_prob[idx]),
                    "y_pred": int(y_prob[idx] >= best_threshold),
                    "selected_threshold_from_validation": float(best_threshold),
                }
            )

    model_path = MODEL_DIR / f"{experiment_id}_{model_family}_best.pt"
    torch.save(
        {
            "experiment_id": experiment_id,
            "config": config,
            "model_state_dict": best_state_dict,
            "best_epoch": best_epoch,
            "best_threshold": best_threshold,
            "feature_names": bundle["train"]["feature_names"].tolist(),
        },
        model_path,
    )

    return {
        "metrics": pd.DataFrame(split_metric_rows),
        "history": pd.DataFrame(history_rows),
        "predictions": pd.DataFrame(prediction_rows),
        "model_path": model_path,
    }


print("Training utilities are ready.")
print("Set RUN_PHASE3A_TRAINING = True in the next cell to train 3 experiments manually.")

Training utilities are ready.
Set RUN_PHASE3A_TRAINING = True in the next cell to train 3 experiments manually.


In [4]:
# Cell 3. Phase 3A 학습 실행
# 원하는 결과:
# - phase3a_000, phase3a_001, phase3a_002 총 3개 실험을 학습한다.
# - validation balanced accuracy 기준 early stopping을 적용한다.
# - validation에서 선택한 threshold로 train/validation/test metrics를 저장한다.
# - metrics/history/predictions/model checkpoint를 저장한다.
#
# 실행 후 보내줄 출력:
# 1. 각 experiment의 early stopping/best epoch 로그
# 2. Final Phase 3A Metrics 표
# 3. Validation Ranking 표
# 4. Test Metrics Paired With Validation Ranking 표

RUN_PHASE3A_TRAINING = True

if not RUN_PHASE3A_TRAINING:
    print("RUN_PHASE3A_TRAINING is False. Training was skipped.")
else:
    all_metrics = []
    all_history = []
    all_predictions = []
    saved_model_paths = []

    print("=== Phase 3A Training Started ===")
    print("device:", DEVICE)
    print("experiments:", phase3a_grid["experiment_id"].tolist())

    for experiment_id in phase3a_grid["experiment_id"].tolist():
        print("\n" + "=" * 80)
        print(f"Training {experiment_id}")
        print("=" * 80)

        result = train_one_experiment(
            experiment_id=experiment_id,
            bundle=loaded_experiments[experiment_id],
            device=DEVICE,
        )

        all_metrics.append(result["metrics"])
        all_history.append(result["history"])
        all_predictions.append(result["predictions"])
        saved_model_paths.append(result["model_path"])

        print(f"saved model: {result['model_path'].relative_to(PROJECT_ROOT)}")

    phase3a_metrics = pd.concat(all_metrics, ignore_index=True)
    phase3a_history = pd.concat(all_history, ignore_index=True)
    phase3a_predictions = pd.concat(all_predictions, ignore_index=True)

    phase3a_metrics.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")
    phase3a_history.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
    phase3a_predictions.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

    print("\n=== Saved Phase 3A Outputs ===")
    print("metrics:", METRICS_PATH.relative_to(PROJECT_ROOT), METRICS_PATH.exists())
    print("history:", HISTORY_PATH.relative_to(PROJECT_ROOT), HISTORY_PATH.exists())
    print("predictions:", PREDICTIONS_PATH.relative_to(PROJECT_ROOT), PREDICTIONS_PATH.exists())
    print("models:")
    for path in saved_model_paths:
        print(" -", path.relative_to(PROJECT_ROOT), path.exists())

    print("\n=== Final Phase 3A Metrics ===")
    display(
        phase3a_metrics
        .sort_values(["experiment_id", "split"])
        .reset_index(drop=True)
    )

    validation_ranking = (
        phase3a_metrics[phase3a_metrics["split"] == "validation"]
        .sort_values(
            ["balanced_accuracy", "roc_auc", "f1"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

    print("\n=== Validation Ranking ===")
    display(validation_ranking)

    test_metrics_paired = (
        validation_ranking[
            [
                "experiment_id",
                "feature_timing",
                "subset_name",
                "model_family",
                "window",
                "best_epoch",
                "selected_threshold_from_validation",
                "balanced_accuracy",
                "roc_auc",
                "average_precision",
                "f1",
                "precision",
                "recall",
            ]
        ]
        .rename(
            columns={
                "balanced_accuracy": "validation_balanced_accuracy",
                "roc_auc": "validation_roc_auc",
                "average_precision": "validation_average_precision",
                "f1": "validation_f1",
                "precision": "validation_precision",
                "recall": "validation_recall",
            }
        )
        .merge(
            phase3a_metrics[phase3a_metrics["split"] == "test"][
                [
                    "experiment_id",
                    "balanced_accuracy",
                    "roc_auc",
                    "average_precision",
                    "f1",
                    "precision",
                    "recall",
                    "tn",
                    "fp",
                    "fn",
                    "tp",
                ]
            ].rename(
                columns={
                    "balanced_accuracy": "test_balanced_accuracy",
                    "roc_auc": "test_roc_auc",
                    "average_precision": "test_average_precision",
                    "f1": "test_f1",
                    "precision": "test_precision",
                    "recall": "test_recall",
                    "tn": "test_tn",
                    "fp": "test_fp",
                    "fn": "test_fn",
                    "tp": "test_tp",
                }
            ),
            on="experiment_id",
            how="left",
        )
    )

    print("\n=== Test Metrics Paired With Validation Ranking ===")
    display(test_metrics_paired)

    print("\nReference previous-day candidate:")
    print("phase2a_006 | previous_day / daytime_only / window 14 / BiLSTM | test balanced accuracy = 0.6098")

=== Phase 3A Training Started ===
device: cpu
experiments: ['phase3a_000', 'phase3a_001', 'phase3a_002']

Training phase3a_000
phase3a_000 epoch 001 | loss=0.8105 | val_bal_acc=0.6041 | val_auc=0.5895 | thr=0.51
phase3a_000 epoch 010 | loss=0.6665 | val_bal_acc=0.5684 | val_auc=0.5514 | thr=0.59
phase3a_000 epoch 020 | loss=0.6059 | val_bal_acc=0.5590 | val_auc=0.4925 | thr=0.72
phase3a_000 early stopped at epoch 21; best_epoch=1, best_val_bal_acc=0.6041
saved model: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\models\phase3a_000_mlp_current_day_best.pt

Training phase3a_001
phase3a_001 epoch 001 | loss=0.7973 | val_bal_acc=0.5758 | val_auc=0.5811 | thr=0.51
phase3a_001 epoch 010 | loss=0.6137 | val_bal_acc=0.6419 | val_auc=0.6566 | thr=0.34
phase3a_001 epoch 020 | loss=0.4551 | val_bal_acc=0.6139 | val_auc=0.5864 | thr=0.52
phase3a_001 early stopped at epoch 23; best_epoch=3, best_val_bal_acc=0.6941
saved model: data\proces

,experiment_id,feature_timing,subset_name,model_family,window,split,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,roc_auc,average_precision,tn,fp,fn,tp
0,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,test,1,0.51,0.51,0.567824,0.453125,0.470270,0.437186,0.612973,0.459533,227,98,112,87
1,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,train,1,0.51,0.51,0.660279,0.611171,0.571285,0.657044,0.715259,0.623663,842,427,297,569
2,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,validation,1,0.51,0.51,0.604081,0.543147,0.500000,0.594444,0.589450,0.452944,170,107,73,107
3,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,test,3,0.35,0.35,0.605386,0.614458,0.488038,0.829268,0.638470,0.525037,66,107,21,102
4,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,train,3,0.35,0.35,0.666667,0.656965,0.515778,0.904580,0.776741,0.692843,334,445,50,474
5,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,validation,3,0.35,0.35,0.694054,0.650206,0.523179,0.858696,0.692313,0.523532,81,72,13,79
6,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,test,4,0.36,0.36,0.579258,0.627615,0.528169,0.773196,0.589804,0.545315,42,67,22,75
7,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,train,4,0.36,0.36,0.661920,0.629118,0.487644,0.886228,0.780652,0.668711,242,311,38,296
8,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,validation,4,0.36,0.36,0.757576,0.689655,0.588235,0.833333,0.739820,0.540329,60,28,8,40



=== Validation Ranking ===


,experiment_id,feature_timing,subset_name,model_family,window,split,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,roc_auc,average_precision,tn,fp,fn,tp
0,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,validation,4,0.36,0.36,0.757576,0.689655,0.588235,0.833333,0.739820,0.540329,60,28,8,40
1,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,validation,3,0.35,0.35,0.694054,0.650206,0.523179,0.858696,0.692313,0.523532,81,72,13,79
2,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,validation,1,0.51,0.51,0.604081,0.543147,0.500000,0.594444,0.589450,0.452944,170,107,73,107



=== Test Metrics Paired With Validation Ranking ===


,experiment_id,feature_timing,subset_name,model_family,window,best_epoch,selected_threshold_from_validation,validation_balanced_accuracy,validation_roc_auc,validation_average_precision,...,test_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,test_tn,test_fp,test_fn,test_tp
0,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,4,0.36,0.757576,0.739820,0.540329,...,0.579258,0.589804,0.545315,0.627615,0.528169,0.773196,42,67,22,75
1,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,3,0.35,0.694054,0.692313,0.523532,...,0.605386,0.638470,0.525037,0.614458,0.488038,0.829268,66,107,21,102
2,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,1,0.51,0.604081,0.589450,0.452944,...,0.567824,0.612973,0.459533,0.453125,0.470270,0.437186,227,98,112,87



Reference previous-day candidate:
phase2a_006 | previous_day / daytime_only / window 14 / BiLSTM | test balanced accuracy = 0.6098


In [5]:
# Cell 4. Phase 3A 결과 요약 및 기존 previous-day best와 비교
# 원하는 결과:
# - validation 기준 ranking과 paired test 결과를 CSV로 저장한다.
# - 기존 previous-day reference phase2a_006과 비교한다.
# - 최종 해석에 필요한 compact summary를 만든다.

PHASE3A_RANKING_PATH = OUTPUT_DIR / "phase_3a_validation_ranked_test_summary.csv"

REFERENCE_PREVIOUS_DAY = {
    "experiment_id": "phase2a_006",
    "feature_timing": "previous_day",
    "subset_name": "daytime_only",
    "model_family": "bilstm",
    "window": 14,
    "test_balanced_accuracy": 0.6098,
    "note": "Previous best conservative previous-day candidate before rolling/trend engineering",
}

phase3a_metrics = pd.read_csv(METRICS_PATH, encoding="utf-8-sig")

validation_ranking = (
    phase3a_metrics[phase3a_metrics["split"] == "validation"]
    .sort_values(
        ["balanced_accuracy", "roc_auc", "f1"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

phase3a_ranked_test_summary = (
    validation_ranking[
        [
            "experiment_id",
            "feature_timing",
            "subset_name",
            "model_family",
            "window",
            "best_epoch",
            "selected_threshold_from_validation",
            "balanced_accuracy",
            "roc_auc",
            "average_precision",
            "f1",
            "precision",
            "recall",
        ]
    ]
    .rename(
        columns={
            "balanced_accuracy": "validation_balanced_accuracy",
            "roc_auc": "validation_roc_auc",
            "average_precision": "validation_average_precision",
            "f1": "validation_f1",
            "precision": "validation_precision",
            "recall": "validation_recall",
        }
    )
    .merge(
        phase3a_metrics[phase3a_metrics["split"] == "test"][
            [
                "experiment_id",
                "balanced_accuracy",
                "roc_auc",
                "average_precision",
                "f1",
                "precision",
                "recall",
                "tn",
                "fp",
                "fn",
                "tp",
            ]
        ].rename(
            columns={
                "balanced_accuracy": "test_balanced_accuracy",
                "roc_auc": "test_roc_auc",
                "average_precision": "test_average_precision",
                "f1": "test_f1",
                "precision": "test_precision",
                "recall": "test_recall",
                "tn": "test_tn",
                "fp": "test_fp",
                "fn": "test_fn",
                "tp": "test_tp",
            }
        ),
        on="experiment_id",
        how="left",
    )
)

phase3a_ranked_test_summary["reference_previous_day_best_balanced_accuracy"] = REFERENCE_PREVIOUS_DAY[
    "test_balanced_accuracy"
]
phase3a_ranked_test_summary["delta_vs_phase2a_006"] = (
    phase3a_ranked_test_summary["test_balanced_accuracy"]
    - REFERENCE_PREVIOUS_DAY["test_balanced_accuracy"]
)

phase3a_ranked_test_summary.to_csv(
    PHASE3A_RANKING_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("=== Phase 3A Ranked Test Summary ===")
display(phase3a_ranked_test_summary)

print("\n=== Reference Previous-Day Best ===")
display(pd.DataFrame([REFERENCE_PREVIOUS_DAY]))

print("\n=== Main Interpretation ===")
best_by_validation = phase3a_ranked_test_summary.iloc[0]
best_by_test = (
    phase3a_ranked_test_summary
    .sort_values("test_balanced_accuracy", ascending=False)
    .iloc[0]
)

print(
    "Validation-selected Phase 3A candidate:",
    best_by_validation["experiment_id"],
    "|",
    best_by_validation["model_family"],
    "| window",
    int(best_by_validation["window"]),
    "| validation balanced accuracy",
    round(float(best_by_validation["validation_balanced_accuracy"]), 4),
    "| test balanced accuracy",
    round(float(best_by_validation["test_balanced_accuracy"]), 4),
)

print(
    "Best Phase 3A test candidate:",
    best_by_test["experiment_id"],
    "|",
    best_by_test["model_family"],
    "| window",
    int(best_by_test["window"]),
    "| test balanced accuracy",
    round(float(best_by_test["test_balanced_accuracy"]), 4),
    "| delta vs phase2a_006",
    round(float(best_by_test["delta_vs_phase2a_006"]), 4),
)

print("saved:", PHASE3A_RANKING_PATH.relative_to(PROJECT_ROOT), PHASE3A_RANKING_PATH.exists())

=== Phase 3A Ranked Test Summary ===


,experiment_id,feature_timing,subset_name,model_family,window,best_epoch,selected_threshold_from_validation,validation_balanced_accuracy,validation_roc_auc,validation_average_precision,...,test_average_precision,test_f1,test_precision,test_recall,test_tn,test_fp,test_fn,test_tp,reference_previous_day_best_balanced_accuracy,delta_vs_phase2a_006
0,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,4,0.36,0.757576,0.739820,0.540329,...,0.545315,0.627615,0.528169,0.773196,42,67,22,75,0.6098,-0.030542
1,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,3,0.35,0.694054,0.692313,0.523532,...,0.525037,0.614458,0.488038,0.829268,66,107,21,102,0.6098,-0.004414
2,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,1,0.51,0.604081,0.589450,0.452944,...,0.459533,0.453125,0.470270,0.437186,227,98,112,87,0.6098,-0.041976



=== Reference Previous-Day Best ===


,experiment_id,feature_timing,subset_name,model_family,window,test_balanced_accuracy,note
0,phase2a_006,previous_day,daytime_only,bilstm,14,0.6098,Previous best conservative previous-day candid...



=== Main Interpretation ===
Validation-selected Phase 3A candidate: phase3a_002 | gru | window 14 | validation balanced accuracy 0.7576 | test balanced accuracy 0.5793
Best Phase 3A test candidate: phase3a_001 | gru | window 7 | test balanced accuracy 0.6054 | delta vs phase2a_006 -0.0044
saved: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\phase_3a_validation_ranked_test_summary.csv True


In [6]:
# Cell 5. Phase 3A training 결과를 work log에 기록한다.
# 원하는 결과:
# - Phase 3A 학습 실행 결과와 해석을 log/2026-06-29.md에 기록한다.
# - 02 pipeline summary notebook은 아직 업데이트하지 않는다.

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

phase3a_log_entry = f"""

## Phase 3A Rolling/Trend Previous-Day Training Run

### 1. Reason for update

- The user manually ran Phase 3A training in `notebooks/09_previous_day_rolling_trend_phase3a_training.ipynb`.
- The goal was to test whether rolling/trend feature engineering improves strict previous-day forecasting.
- Training used the saved `daytime_only_rolling_trend` tensors.
- Logistic Regression and Random Forest remain traditional ML baseline/reference only.
- PCA was not used.

### 2. Created outputs

- Created:
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_rolling_trend_model_metrics.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_rolling_trend_training_history.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_rolling_trend_model_predictions.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_validation_ranked_test_summary.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/models/phase3a_000_mlp_current_day_best.pt`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/models/phase3a_001_gru_best.pt`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/models/phase3a_002_gru_best.pt`

### 3. Phase 3A validation ranking

| experiment_id | model | window | best_epoch | validation balanced accuracy | test balanced accuracy | test ROC AUC |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| `phase3a_002` | `GRU` | 14 | 4 | 0.7576 | 0.5793 | 0.5898 |
| `phase3a_001` | `GRU` | 7 | 3 | 0.6941 | 0.6054 | 0.6385 |
| `phase3a_000` | `MLP current-day` | 1 | 1 | 0.6041 | 0.5678 | 0.6130 |

### 4. Interpretation

- Validation-selected Phase 3A candidate:
  - `phase3a_002`
  - previous-day `daytime_only_rolling_trend`
  - GRU window 14
  - validation balanced accuracy: `0.7576`
  - test balanced accuracy: `0.5793`
- Best Phase 3A test candidate:
  - `phase3a_001`
  - previous-day `daytime_only_rolling_trend`
  - GRU window 7
  - test balanced accuracy: `0.6054`
- Previous best conservative previous-day candidate remains:
  - `phase2a_006`
  - previous-day `daytime_only`
  - BiLSTM window 14
  - test balanced accuracy: `0.6098`

### 5. Current conclusion

- Rolling/trend feature engineering improved validation performance for sequence models but did not clearly beat the existing previous-day reference candidate on held-out test balanced accuracy.
- Strict previous-day final model should not be replaced based on Phase 3A alone.
- The current previous-day reference remains `phase2a_006`, while `phase3a_001` is a close rolling/trend comparison candidate.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(phase3a_log_entry)

print("log updated:", LOG_PATH)
print("exists:", LOG_PATH.exists())
print("appended characters:", len(phase3a_log_entry))

log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md
exists: True
appended characters: 2956


In [ ]:
# Cell 6. Phase 3A threshold sensitivity 분석
# 원하는 결과:
# - 저장된 prediction을 사용해서 threshold별 metric을 다시 계산한다.
# - validation balanced accuracy 최적 threshold와 validation F1 최적 threshold를 비교한다.
# - 각 threshold 선택이 test precision/recall/F1/balanced accuracy에 주는 영향을 확인한다.
# - training은 재실행하지 않는다.

THRESHOLD_SENSITIVITY_PATH = OUTPUT_DIR / "phase_3a_threshold_sensitivity.csv"
THRESHOLD_POLICY_COMPARISON_PATH = OUTPUT_DIR / "phase_3a_threshold_policy_comparison.csv"

phase3a_predictions = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")
phase3a_metrics = pd.read_csv(METRICS_PATH, encoding="utf-8-sig")

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

threshold_rows = []

for experiment_id, exp_df in phase3a_predictions.groupby("experiment_id"):
    for split, split_df in exp_df.groupby("split"):
        y_true = split_df["y_true"].to_numpy(dtype=int)
        y_prob = split_df["y_prob"].to_numpy(dtype=float)

        for threshold in thresholds:
            metric = binary_metrics(y_true, y_prob, threshold=threshold)
            threshold_rows.append(
                {
                    "experiment_id": experiment_id,
                    "split": split,
                    **metric,
                }
            )

threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_sensitivity.to_csv(
    THRESHOLD_SENSITIVITY_PATH,
    index=False,
    encoding="utf-8-sig",
)

policy_rows = []

for experiment_id in sorted(phase3a_predictions["experiment_id"].unique()):
    validation_thresholds = threshold_sensitivity[
        (threshold_sensitivity["experiment_id"] == experiment_id)
        & (threshold_sensitivity["split"] == "validation")
    ].copy()

    selected_by_balanced_accuracy = (
        validation_thresholds
        .sort_values(
            ["balanced_accuracy", "f1", "recall"],
            ascending=[False, False, False],
        )
        .iloc[0]
    )

    selected_by_f1 = (
        validation_thresholds
        .sort_values(
            ["f1", "balanced_accuracy", "precision"],
            ascending=[False, False, False],
        )
        .iloc[0]
    )

    selected_by_precision_recall_balance = validation_thresholds.copy()
    selected_by_precision_recall_balance["precision_recall_gap"] = (
        selected_by_precision_recall_balance["precision"]
        - selected_by_precision_recall_balance["recall"]
    ).abs()

    selected_by_precision_recall_balance = (
        selected_by_precision_recall_balance
        .sort_values(
            ["f1", "precision_recall_gap", "balanced_accuracy"],
            ascending=[False, True, False],
        )
        .iloc[0]
    )

    policies = {
        "validation_balanced_accuracy": selected_by_balanced_accuracy,
        "validation_f1": selected_by_f1,
        "validation_f1_with_precision_recall_balance": selected_by_precision_recall_balance,
    }

    for policy_name, selected_row in policies.items():
        selected_threshold = float(selected_row["threshold"])

        for split in ["validation", "test"]:
            metric_row = threshold_sensitivity[
                (threshold_sensitivity["experiment_id"] == experiment_id)
                & (threshold_sensitivity["split"] == split)
                & (threshold_sensitivity["threshold"] == selected_threshold)
            ].iloc[0]

            policy_rows.append(
                {
                    "experiment_id": experiment_id,
                    "threshold_policy": policy_name,
                    "selected_threshold": selected_threshold,
                    "evaluation_split": split,
                    "balanced_accuracy": metric_row["balanced_accuracy"],
                    "roc_auc": metric_row["roc_auc"],
                    "average_precision": metric_row["average_precision"],
                    "f1": metric_row["f1"],
                    "precision": metric_row["precision"],
                    "recall": metric_row["recall"],
                    "tn": int(metric_row["tn"]),
                    "fp": int(metric_row["fp"]),
                    "fn": int(metric_row["fn"]),
                    "tp": int(metric_row["tp"]),
                }
            )

threshold_policy_comparison = pd.DataFrame(policy_rows)
threshold_policy_comparison.to_csv(
    THRESHOLD_POLICY_COMPARISON_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("=== Threshold Sensitivity Saved ===")
print("threshold sensitivity:", THRESHOLD_SENSITIVITY_PATH.relative_to(PROJECT_ROOT), THRESHOLD_SENSITIVITY_PATH.exists())
print("policy comparison:", THRESHOLD_POLICY_COMPARISON_PATH.relative_to(PROJECT_ROOT), THRESHOLD_POLICY_COMPARISON_PATH.exists())

print("\n=== Threshold Policy Comparison: Validation Rows ===")
display(
    threshold_policy_comparison[
        threshold_policy_comparison["evaluation_split"] == "validation"
    ]
    .sort_values(["experiment_id", "threshold_policy"])
    .reset_index(drop=True)
)

print("\n=== Threshold Policy Comparison: Test Rows ===")
display(
    threshold_policy_comparison[
        threshold_policy_comparison["evaluation_split"] == "test"
    ]
    .sort_values(["experiment_id", "threshold_policy"])
    .reset_index(drop=True)
)

print("\n=== Best Test F1 Per Experiment Across All Thresholds (diagnostic only) ===")
best_test_f1_diagnostic = (
    threshold_sensitivity[threshold_sensitivity["split"] == "test"]
    .sort_values(["experiment_id", "f1", "balanced_accuracy"], ascending=[True, False, False])
    .groupby("experiment_id")
    .head(1)
    .reset_index(drop=True)
)

display(best_test_f1_diagnostic)

=== Threshold Sensitivity Saved ===
threshold sensitivity: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\phase_3a_threshold_sensitivity.csv True
policy comparison: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\phase_3a_threshold_policy_comparison.csv True

=== Threshold Policy Comparison: Validation Rows ===


,experiment_id,threshold_policy,selected_threshold,evaluation_split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
0,phase3a_000,validation_balanced_accuracy,0.51,validation,0.604081,0.589450,0.452944,0.543147,0.500000,0.594444,170,107,73,107
1,phase3a_000,validation_f1,0.38,validation,0.534838,0.589450,0.452944,0.566434,0.413265,0.900000,47,230,18,162
2,phase3a_000,validation_f1_with_precision_recall_balance,0.38,validation,0.534838,0.589450,0.452944,0.566434,0.413265,0.900000,47,230,18,162
3,phase3a_001,validation_balanced_accuracy,0.35,validation,0.694054,0.692313,0.523532,0.650206,0.523179,0.858696,81,72,13,79
4,phase3a_001,validation_f1,0.35,validation,0.694054,0.692313,0.523532,0.650206,0.523179,0.858696,81,72,13,79
5,phase3a_001,validation_f1_with_precision_recall_balance,0.35,validation,0.694054,0.692313,0.523532,0.650206,0.523179,0.858696,81,72,13,79
6,phase3a_002,validation_balanced_accuracy,0.36,validation,0.757576,0.739820,0.540329,0.689655,0.588235,0.833333,60,28,8,40
7,phase3a_002,validation_f1,0.36,validation,0.757576,0.739820,0.540329,0.689655,0.588235,0.833333,60,28,8,40
8,phase3a_002,validation_f1_with_precision_recall_balance,0.36,validation,0.757576,0.739820,0.540329,0.689655,0.588235,0.833333,60,28,8,40



=== Threshold Policy Comparison: Test Rows ===


,experiment_id,threshold_policy,selected_threshold,evaluation_split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
0,phase3a_000,validation_balanced_accuracy,0.51,test,0.567824,0.612973,0.459533,0.453125,0.470270,0.437186,227,98,112,87
1,phase3a_000,validation_f1,0.38,test,0.568775,0.612973,0.459533,0.571429,0.419811,0.894472,79,246,21,178
2,phase3a_000,validation_f1_with_precision_recall_balance,0.38,test,0.568775,0.612973,0.459533,0.571429,0.419811,0.894472,79,246,21,178
3,phase3a_001,validation_balanced_accuracy,0.35,test,0.605386,0.638470,0.525037,0.614458,0.488038,0.829268,66,107,21,102
4,phase3a_001,validation_f1,0.35,test,0.605386,0.638470,0.525037,0.614458,0.488038,0.829268,66,107,21,102
5,phase3a_001,validation_f1_with_precision_recall_balance,0.35,test,0.605386,0.638470,0.525037,0.614458,0.488038,0.829268,66,107,21,102
6,phase3a_002,validation_balanced_accuracy,0.36,test,0.579258,0.589804,0.545315,0.627615,0.528169,0.773196,42,67,22,75
7,phase3a_002,validation_f1,0.36,test,0.579258,0.589804,0.545315,0.627615,0.528169,0.773196,42,67,22,75
8,phase3a_002,validation_f1_with_precision_recall_balance,0.36,test,0.579258,0.589804,0.545315,0.627615,0.528169,0.773196,42,67,22,75



=== Best Test F1 Per Experiment Across All Thresholds (diagnostic only) ===


,experiment_id,split,threshold,balanced_accuracy,f1,precision,recall,roc_auc,average_precision,tn,fp,fn,tp
0,phase3a_000,test,0.40,0.590622,0.580101,0.436548,0.864322,0.612973,0.459533,103,222,27,172
1,phase3a_001,test,0.33,0.618850,0.636103,0.491150,0.902439,0.638470,0.525037,58,115,12,111
2,phase3a_002,test,0.19,0.513761,0.646667,0.477833,1.000000,0.589804,0.545315,3,106,0,97


In [8]:
# Cell 7. Phase 3A threshold sensitivity 결과를 log에 기록한다.
# 원하는 결과:
# - threshold sensitivity 파일 생성과 주요 해석을 log/2026-06-29.md에 기록한다.
# - test-optimized threshold는 diagnostic only라고 명시한다.

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

threshold_log_entry = """

## Phase 3A Threshold Sensitivity Analysis

### 1. Reason for update

- Ran threshold sensitivity analysis using saved Phase 3A prediction probabilities.
- Training was not rerun.
- The goal was to check whether low F1/precision was mainly due to validation-selected threshold choice.

### 2. Created outputs

- Created:
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_threshold_sensitivity.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_threshold_policy_comparison.csv`

### 3. Main findings

- For `phase3a_001` and `phase3a_002`, the validation-balanced-accuracy threshold and validation-F1 threshold were identical:
  - `phase3a_001`: threshold `0.35`
  - `phase3a_002`: threshold `0.36`
- Therefore, switching from validation balanced accuracy to validation F1 did not materially change official test metrics for the GRU candidates.
- The best validation-selected Phase 3A test candidate remained:
  - `phase3a_001`
  - previous-day `daytime_only_rolling_trend`
  - GRU window 7
  - test balanced accuracy `0.6054`
  - test F1 `0.6145`
  - test precision `0.4880`
  - test recall `0.8293`
- A diagnostic test-threshold sweep showed that `phase3a_001` could reach test balanced accuracy around `0.6189` and F1 around `0.6361` at threshold `0.33`.
- This diagnostic test-optimized threshold must not be used for final model selection because it uses held-out test labels.
- `phase3a_002` could reach high diagnostic test F1 only by using an almost all-positive threshold, with very poor specificity.

### 4. Current interpretation

- Phase 3A rolling/trend features make the GRU window-7 candidate close to the existing previous-day reference, but do not clearly beat it under validation-selected thresholding.
- Precision remains limited and false positives remain a major weakness.
- The strict previous-day final model should not be replaced based on Phase 3A alone.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(threshold_log_entry)

print("log updated:", LOG_PATH)
print("exists:", LOG_PATH.exists())
print("appended characters:", len(threshold_log_entry))

log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md
exists: True
appended characters: 2027


In [9]:
# Cell 8. Phase 3A rolling/trend 결과 보고서 생성
# 원하는 결과:
# - reports/phase3a_previous_day_rolling_trend_report.md 파일을 생성한다.
# - Phase 3A feature engineering, training, threshold sensitivity 결과를 정리한다.
# - strict previous-day 최종 모델 교체 여부를 명확히 기록한다.
# - 생성 후 log/2026-06-29.md도 업데이트한다.

REPORT_PATH = PROJECT_ROOT / "reports" / "phase3a_previous_day_rolling_trend_report.md"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

phase3a_ranked_test_summary = pd.read_csv(
    OUTPUT_DIR / "phase_3a_validation_ranked_test_summary.csv",
    encoding="utf-8-sig",
)

threshold_policy_comparison = pd.read_csv(
    OUTPUT_DIR / "phase_3a_threshold_policy_comparison.csv",
    encoding="utf-8-sig",
)

tensor_summary = pd.read_csv(
    ROLLING_TREND_DIR / "tensor_summary.csv",
    encoding="utf-8-sig",
)

validation_rows = threshold_policy_comparison[
    threshold_policy_comparison["evaluation_split"] == "validation"
].copy()

test_rows = threshold_policy_comparison[
    threshold_policy_comparison["evaluation_split"] == "test"
].copy()

report_text = f"""# Phase 3A Previous-Day Rolling/Trend Feature Engineering Report

## Summary

Phase 3A tested whether rolling and trend features improve strict previous-day sleep forecasting.

The experiment used the previous-day dataset and started from the conservative `daytime_only` feature subset. Rolling and trend features were created within participant-level contiguous date blocks so that feature histories did not cross date gaps.

The main result is that rolling/trend features made the GRU window-7 candidate close to the previous-day reference candidate, but did not clearly exceed it under validation-selected thresholding.

## Inputs

- Previous-day encoded dataset:
  - `data/processed/deep_learning_previous_day/modeling_dataset_previous_day_encoded.csv`
- Base feature subset:
  - `data/processed/deep_learning_feature_subsets/daytime_only_features.csv`
- Phase 3A experiment grid:
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_rolling_trend_experiment_grid.csv`

## Feature Engineering

The source rows were strict previous-day rows where `feature_date = target calendar_date - 1 day`.

Feature engineering used the 19 `daytime_only` features and created:

- 3-day and 7-day rolling mean
- 3-day and 7-day rolling standard deviation
- 3-day and 7-day rolling minimum
- 3-day and 7-day rolling maximum
- within-block 1-row difference
- deviation from 3-day rolling mean
- deviation from 7-day rolling mean

Rolling history was computed within participant-level contiguous date blocks. A new block was started whenever the `feature_date` gap was not exactly 1 day.

Calendar features were prepared as optional candidates but were not included in this saved Phase 3A tensor set.

## Saved Feature Set

- Base daytime-only features: 19
- Engineered candidate features before filtering: 228
- Train zero-variance features removed: 106
- Final saved feature count: 122

Saved feature-engineering outputs:

- `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/modeling_dataset_previous_day_daytime_only_rolling_trend.csv`
- `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/rolling_trend_daytime_only_feature_columns.csv`
- `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/zero_variance_removed_features.csv`
- `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/rolling_trend_daytime_only_standard_scaler.joblib`
- `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/metadata.json`
- `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/tensor_summary.csv`

## Tensor Summary

| tensor_type | window | split | samples | participants | features | target_mean |
| --- | ---: | --- | ---: | ---: | ---: | ---: |
"""

for _, row in tensor_summary.sort_values(["tensor_type", "window", "split"]).iterrows():
    report_text += (
        f"| `{row['tensor_type']}` | {int(row['window'])} | `{row['split']}` | "
        f"{int(row['samples'])} | {int(row['participants'])} | {int(row['features'])} | "
        f"{float(row['target_mean']):.4f} |\n"
    )

report_text += """

## Phase 3A Experiments

| experiment_id | feature_timing | subset_name | model_family | window |
| --- | --- | --- | --- | ---: |
| `phase3a_000` | previous_day | `daytime_only_rolling_trend` | `mlp_current_day` | 1 |
| `phase3a_001` | previous_day | `daytime_only_rolling_trend` | `gru` | 7 |
| `phase3a_002` | previous_day | `daytime_only_rolling_trend` | `gru` | 14 |

Training used validation balanced accuracy for early stopping and threshold selection. Test metrics were then computed using the validation-selected threshold.

## Validation Ranking And Paired Test Metrics

| experiment_id | model | window | best_epoch | threshold | validation balanced accuracy | validation ROC AUC | test balanced accuracy | test ROC AUC | test F1 | test precision | test recall | delta vs phase2a_006 |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
"""

for _, row in phase3a_ranked_test_summary.iterrows():
    report_text += (
        f"| `{row['experiment_id']}` | `{row['model_family']}` | {int(row['window'])} | "
        f"{int(row['best_epoch'])} | {float(row['selected_threshold_from_validation']):.2f} | "
        f"{float(row['validation_balanced_accuracy']):.4f} | {float(row['validation_roc_auc']):.4f} | "
        f"{float(row['test_balanced_accuracy']):.4f} | {float(row['test_roc_auc']):.4f} | "
        f"{float(row['test_f1']):.4f} | {float(row['test_precision']):.4f} | "
        f"{float(row['test_recall']):.4f} | {float(row['delta_vs_phase2a_006']):.4f} |\n"
    )

report_text += """

## Comparison To Previous-Day Reference

The previous best conservative previous-day candidate was:

| experiment_id | feature_timing | subset_name | model_family | window | test balanced accuracy |
| --- | --- | --- | --- | ---: | ---: |
| `phase2a_006` | previous_day | `daytime_only` | `BiLSTM` | 14 | 0.6098 |

The best Phase 3A test candidate under validation-selected thresholding was:

| experiment_id | model | window | test balanced accuracy | test ROC AUC | test F1 | test precision | test recall |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
"""

best_phase3a_test = (
    phase3a_ranked_test_summary
    .sort_values("test_balanced_accuracy", ascending=False)
    .iloc[0]
)

report_text += (
    f"| `{best_phase3a_test['experiment_id']}` | `{best_phase3a_test['model_family']}` | "
    f"{int(best_phase3a_test['window'])} | "
    f"{float(best_phase3a_test['test_balanced_accuracy']):.4f} | "
    f"{float(best_phase3a_test['test_roc_auc']):.4f} | "
    f"{float(best_phase3a_test['test_f1']):.4f} | "
    f"{float(best_phase3a_test['test_precision']):.4f} | "
    f"{float(best_phase3a_test['test_recall']):.4f} |\n"
)

report_text += """

This result was close to `phase2a_006`, but did not clearly exceed it.

## Threshold Sensitivity

Threshold sensitivity analysis used saved prediction probabilities only. Training was not rerun.

For `phase3a_001` and `phase3a_002`, the validation balanced-accuracy threshold and validation F1 threshold were identical. This means switching from balanced accuracy to F1 as the validation threshold policy did not materially change the official test metrics for the GRU candidates.

| experiment_id | threshold policy | threshold | split | balanced accuracy | F1 | precision | recall |
| --- | --- | ---: | --- | ---: | ---: | ---: | ---: |
"""

threshold_display = threshold_policy_comparison[
    threshold_policy_comparison["threshold_policy"].isin(
        ["validation_balanced_accuracy", "validation_f1"]
    )
].copy()

for _, row in threshold_display.sort_values(
    ["experiment_id", "threshold_policy", "evaluation_split"]
).iterrows():
    report_text += (
        f"| `{row['experiment_id']}` | `{row['threshold_policy']}` | "
        f"{float(row['selected_threshold']):.2f} | `{row['evaluation_split']}` | "
        f"{float(row['balanced_accuracy']):.4f} | {float(row['f1']):.4f} | "
        f"{float(row['precision']):.4f} | {float(row['recall']):.4f} |\n"
    )

report_text += """

A diagnostic test-threshold sweep suggested that `phase3a_001` could reach test balanced accuracy around 0.6189 and F1 around 0.6361 at threshold 0.33. However, this threshold was selected using held-out test labels, so it must not be used for final model selection.

For `phase3a_002`, the highest diagnostic test F1 was achieved by an almost all-positive threshold, which produced very poor specificity. This supports keeping validation-selected thresholding as the official comparison rule.

## Interpretation

Phase 3A shows that rolling/trend feature engineering can improve validation performance for sequence models, especially GRU window 14. However, the validation-selected candidate did not transfer as strongly to the held-out test split.

The most useful Phase 3A comparison candidate is:

`previous_day / daytime_only_rolling_trend / GRU / window 7`

It reached test balanced accuracy close to the previous-day reference candidate, but did not clearly exceed it.

Precision remained limited across Phase 3A GRU candidates. The models achieved relatively high recall but produced many false positives.

## Conclusion

Phase 3A does not justify replacing the current previous-day reference candidate.

Current previous-day reference candidate:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

Close rolling/trend comparison candidate:

`phase3a_001 / previous_day / daytime_only_rolling_trend / window 7 / GRU`

The current best overall model remains the same-date classifier:

`phase1a_003 / same_date / daytime_only / window 7 / mlp_current_day`

That same-date model should not be described as strict previous-day forecasting.
"""

REPORT_PATH.write_text(report_text, encoding="utf-8")

log_entry = f"""

## Phase 3A Rolling/Trend Report

### 1. Reason for update

- Created a report summarizing Phase 3A previous-day rolling/trend feature engineering, training, and threshold sensitivity.
- The report records that Phase 3A does not clearly replace the current previous-day reference candidate.

### 2. Created report

- Created:
  - `reports/phase3a_previous_day_rolling_trend_report.md`

### 3. Main conclusion

- Best validation-selected Phase 3A candidate:
  - `phase3a_002`
  - previous-day `daytime_only_rolling_trend`
  - GRU window 14
  - validation balanced accuracy `0.7576`
  - test balanced accuracy `0.5793`
- Best Phase 3A test candidate:
  - `phase3a_001`
  - previous-day `daytime_only_rolling_trend`
  - GRU window 7
  - test balanced accuracy `0.6054`
- Current previous-day reference remains:
  - `phase2a_006`
  - previous-day `daytime_only`
  - BiLSTM window 14
  - test balanced accuracy `0.6098`
- Phase 3A is useful as a close rolling/trend comparison but does not justify replacing the previous-day reference candidate.

### 4. Validation

- Wrote report as UTF-8 markdown.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("report written:", REPORT_PATH.relative_to(PROJECT_ROOT), REPORT_PATH.exists())
print("report characters:", len(report_text))
print("log updated:", LOG_PATH, LOG_PATH.exists())
print("log appended characters:", len(log_entry))

report written: reports\phase3a_previous_day_rolling_trend_report.md True
report characters: 8703
log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
log appended characters: 1097


In [12]:
# Cell 9. phase2a_006 vs phase3a_001 직접 비교 요약 생성
# 원하는 결과:
# - 기존 previous-day reference와 Phase 3A best test candidate를 한 표로 비교한다.
# - 왜 previous-day reference를 유지하는지 짧게 정리한다.
# - CSV/Markdown을 저장하고 log를 업데이트한다.

REFERENCE_COMPARISON_CSV_PATH = OUTPUT_DIR / "phase2a_006_vs_phase3a_001_direct_comparison.csv"
REFERENCE_COMPARISON_MD_PATH = OUTPUT_DIR / "phase2a_006_vs_phase3a_001_direct_comparison.md"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

# Phase 2A의 기존 metrics 파일에서 phase2a_006 test row를 직접 불러옵니다.
PHASE2A_METRICS_PATH = (
    PREVIOUS_DAY_DIR
    / "experiments"
    / "phase_2a_outputs"
    / "phase_2a_previous_day_model_metrics.csv"
)

phase2a_metrics = pd.read_csv(PHASE2A_METRICS_PATH, encoding="utf-8-sig")
phase3a_ranked_test_summary = pd.read_csv(
    OUTPUT_DIR / "phase_3a_validation_ranked_test_summary.csv",
    encoding="utf-8-sig",
)

phase2a_reference = phase2a_metrics[
    (phase2a_metrics["experiment_id"] == "phase2a_006")
    & (phase2a_metrics["split"] == "test")
].iloc[0]

phase3a_candidate = phase3a_ranked_test_summary[
    phase3a_ranked_test_summary["experiment_id"] == "phase3a_001"
].iloc[0]

comparison_rows = [
    {
        "candidate_role": "current_previous_day_reference",
        "experiment_id": "phase2a_006",
        "feature_timing": "previous_day",
        "subset_name": phase2a_reference.get("subset_name", "daytime_only"),
        "model_family": phase2a_reference["model_family"],
        "window": int(phase2a_reference["window"]),
        "selection_basis": "previous Phase 2A validation/test review",
        "test_balanced_accuracy": float(phase2a_reference["balanced_accuracy"]),
        "test_roc_auc": float(phase2a_reference["roc_auc"]),
        "test_average_precision": float(phase2a_reference["average_precision"]),
        "test_f1": float(phase2a_reference["f1"]),
        "test_precision": float(phase2a_reference["precision"]),
        "test_recall": float(phase2a_reference["recall"]),
    },
    {
        "candidate_role": "phase3a_close_rolling_trend_candidate",
        "experiment_id": "phase3a_001",
        "feature_timing": "previous_day",
        "subset_name": "daytime_only_rolling_trend",
        "model_family": phase3a_candidate["model_family"],
        "window": int(phase3a_candidate["window"]),
        "selection_basis": "best Phase 3A held-out test balanced accuracy under validation-selected threshold",
        "test_balanced_accuracy": float(phase3a_candidate["test_balanced_accuracy"]),
        "test_roc_auc": float(phase3a_candidate["test_roc_auc"]),
        "test_average_precision": float(phase3a_candidate["test_average_precision"]),
        "test_f1": float(phase3a_candidate["test_f1"]),
        "test_precision": float(phase3a_candidate["test_precision"]),
        "test_recall": float(phase3a_candidate["test_recall"]),
    },
]

direct_comparison = pd.DataFrame(comparison_rows)
direct_comparison["delta_balanced_accuracy_vs_phase2a_006"] = (
    direct_comparison["test_balanced_accuracy"]
    - direct_comparison.loc[
        direct_comparison["experiment_id"] == "phase2a_006",
        "test_balanced_accuracy",
    ].iloc[0]
)

direct_comparison.to_csv(
    REFERENCE_COMPARISON_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

comparison_md = """# Phase 2A Reference Vs Phase 3A Rolling/Trend Candidate

## Purpose

This file directly compares the current strict previous-day reference candidate with the closest Phase 3A rolling/trend candidate.

## Direct Comparison

| role | experiment_id | subset | model | window | test balanced accuracy | test ROC AUC | test F1 | test precision | test recall | delta balanced accuracy |
| --- | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
"""

for _, row in direct_comparison.iterrows():
    comparison_md += (
        f"| `{row['candidate_role']}` | `{row['experiment_id']}` | `{row['subset_name']}` | "
        f"`{row['model_family']}` | {int(row['window'])} | "
        f"{float(row['test_balanced_accuracy']):.4f} | "
        f"{float(row['test_roc_auc']):.4f} | "
        f"{float(row['test_f1']):.4f} | "
        f"{float(row['test_precision']):.4f} | "
        f"{float(row['test_recall']):.4f} | "
        f"{float(row['delta_balanced_accuracy_vs_phase2a_006']):.4f} |\n"
    )

comparison_md += """

## Conclusion

`phase3a_001` is a close rolling/trend comparison candidate, but it does not clearly exceed `phase2a_006` under validation-selected thresholding.

Therefore, the strict previous-day reference candidate remains:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

The Phase 3A result should be reported as a useful feature-engineering sensitivity analysis rather than a replacement final model.
"""

REFERENCE_COMPARISON_MD_PATH.write_text(comparison_md, encoding="utf-8")

log_entry = f"""

## Phase 2A Reference Vs Phase 3A Candidate Direct Comparison

### 1. Reason for update

- Created a compact direct comparison between the current previous-day reference candidate and the closest Phase 3A rolling/trend candidate.
- This supports the decision to keep `phase2a_006` as the strict previous-day reference.

### 2. Created outputs

- Created:
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase2a_006_vs_phase3a_001_direct_comparison.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase2a_006_vs_phase3a_001_direct_comparison.md`

### 3. Main conclusion

- `phase2a_006` remains the current strict previous-day reference candidate.
- `phase3a_001` is a close rolling/trend comparison candidate but does not clearly exceed `phase2a_006` under validation-selected thresholding.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("comparison csv:", REFERENCE_COMPARISON_CSV_PATH.relative_to(PROJECT_ROOT), REFERENCE_COMPARISON_CSV_PATH.exists())
print("comparison md:", REFERENCE_COMPARISON_MD_PATH.relative_to(PROJECT_ROOT), REFERENCE_COMPARISON_MD_PATH.exists())
print("\n=== Direct Comparison ===")
display(direct_comparison)
print("\nlog updated:", LOG_PATH, LOG_PATH.exists())
print("log appended characters:", len(log_entry))

comparison csv: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\phase2a_006_vs_phase3a_001_direct_comparison.csv True
comparison md: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\phase2a_006_vs_phase3a_001_direct_comparison.md True

=== Direct Comparison ===


,candidate_role,experiment_id,feature_timing,subset_name,model_family,window,selection_basis,test_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,delta_balanced_accuracy_vs_phase2a_006
0,current_previous_day_reference,phase2a_006,previous_day,daytime_only,bilstm,14,previous Phase 2A validation/test review,0.586825,0.606261,0.578799,0.568528,0.560000,0.577320,0.000000
1,phase3a_close_rolling_trend_candidate,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,best Phase 3A held-out test balanced accuracy ...,0.605386,0.638470,0.525037,0.614458,0.488038,0.829268,0.018561



log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
log appended characters: 926


In [13]:
# Cell 10. phase2a_006 vs phase3a_001 직접 비교표 수정
# 원하는 결과:
# - phase2a_006은 bootstrap summary의 validation-selected threshold 0.51 기준 original_value를 사용한다.
# - 이전에 threshold=0.5 metrics 파일에서 읽은 0.5868 값을 공식 reference로 쓰지 않는다.
# - direct comparison CSV/Markdown을 올바른 값으로 덮어쓴다.
# - correction log를 추가한다.

REFERENCE_COMPARISON_CSV_PATH = OUTPUT_DIR / "phase2a_006_vs_phase3a_001_direct_comparison.csv"
REFERENCE_COMPARISON_MD_PATH = OUTPUT_DIR / "phase2a_006_vs_phase3a_001_direct_comparison.md"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

BOOTSTRAP_SUMMARY_PATH = (
    PREVIOUS_DAY_DIR
    / "experiments"
    / "phase_2a_outputs"
    / "selected_candidate_participant_bootstrap_summary.csv"
)

bootstrap_summary = pd.read_csv(BOOTSTRAP_SUMMARY_PATH, encoding="utf-8-sig")
phase3a_ranked_test_summary = pd.read_csv(
    OUTPUT_DIR / "phase_3a_validation_ranked_test_summary.csv",
    encoding="utf-8-sig",
)

phase2a_boot = bootstrap_summary[
    bootstrap_summary["experiment_id"].eq("phase2a_006")
].copy()

phase2a_metric_values = (
    phase2a_boot
    .set_index("metric")["original_value"]
    .to_dict()
)

phase3a_candidate = phase3a_ranked_test_summary[
    phase3a_ranked_test_summary["experiment_id"].eq("phase3a_001")
].iloc[0]

phase2a_bal_acc = float(phase2a_metric_values["balanced_accuracy"])

direct_comparison = pd.DataFrame(
    [
        {
            "candidate_role": "current_previous_day_reference",
            "experiment_id": "phase2a_006",
            "feature_timing": "previous_day",
            "subset_name": "daytime_only",
            "model_family": "bilstm",
            "window": 14,
            "selection_basis": "validation-selected threshold recorded in bootstrap summary",
            "selected_threshold": 0.51,
            "test_balanced_accuracy": phase2a_bal_acc,
            "test_roc_auc": float(phase2a_metric_values["roc_auc"]),
            "test_average_precision": float(phase2a_metric_values["average_precision"]),
            "test_f1": float(phase2a_metric_values["f1"]),
            "test_precision": float(phase2a_metric_values["precision"]),
            "test_recall": float(phase2a_metric_values["recall"]),
        },
        {
            "candidate_role": "phase3a_close_rolling_trend_candidate",
            "experiment_id": "phase3a_001",
            "feature_timing": "previous_day",
            "subset_name": "daytime_only_rolling_trend",
            "model_family": phase3a_candidate["model_family"],
            "window": int(phase3a_candidate["window"]),
            "selection_basis": "Phase 3A validation-selected threshold",
            "selected_threshold": float(phase3a_candidate["selected_threshold_from_validation"]),
            "test_balanced_accuracy": float(phase3a_candidate["test_balanced_accuracy"]),
            "test_roc_auc": float(phase3a_candidate["test_roc_auc"]),
            "test_average_precision": float(phase3a_candidate["test_average_precision"]),
            "test_f1": float(phase3a_candidate["test_f1"]),
            "test_precision": float(phase3a_candidate["test_precision"]),
            "test_recall": float(phase3a_candidate["test_recall"]),
        },
    ]
)

direct_comparison["delta_balanced_accuracy_vs_phase2a_006"] = (
    direct_comparison["test_balanced_accuracy"] - phase2a_bal_acc
)

direct_comparison.to_csv(
    REFERENCE_COMPARISON_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

comparison_md = """# Phase 2A Reference Vs Phase 3A Rolling/Trend Candidate

## Purpose

This file directly compares the current strict previous-day reference candidate with the closest Phase 3A rolling/trend candidate.

Important correction: `phase2a_006` is compared using the validation-selected threshold value recorded in the participant-bootstrap summary, not the raw threshold-0.5 row from `phase_2a_previous_day_model_metrics.csv`.

## Direct Comparison

| role | experiment_id | subset | model | window | threshold | test balanced accuracy | test ROC AUC | test F1 | test precision | test recall | delta balanced accuracy |
| --- | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
"""

for _, row in direct_comparison.iterrows():
    comparison_md += (
        f"| `{row['candidate_role']}` | `{row['experiment_id']}` | `{row['subset_name']}` | "
        f"`{row['model_family']}` | {int(row['window'])} | "
        f"{float(row['selected_threshold']):.2f} | "
        f"{float(row['test_balanced_accuracy']):.4f} | "
        f"{float(row['test_roc_auc']):.4f} | "
        f"{float(row['test_f1']):.4f} | "
        f"{float(row['test_precision']):.4f} | "
        f"{float(row['test_recall']):.4f} | "
        f"{float(row['delta_balanced_accuracy_vs_phase2a_006']):.4f} |\n"
    )

comparison_md += """

## Conclusion

`phase3a_001` is a close rolling/trend comparison candidate, but it does not clearly exceed `phase2a_006` under validation-selected thresholding.

Therefore, the strict previous-day reference candidate remains:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

The Phase 3A result should be reported as a useful feature-engineering sensitivity analysis rather than a replacement final model.
"""

REFERENCE_COMPARISON_MD_PATH.write_text(comparison_md, encoding="utf-8")

log_entry = """

## Direct Comparison Metric Correction

### 1. Reason for update

- Corrected the direct comparison between `phase2a_006` and `phase3a_001`.
- The first comparison accidentally used the raw `phase_2a_previous_day_model_metrics.csv` test row at threshold `0.5`, which gave balanced accuracy `0.5868`.
- The official previous-day reference value is the validation-selected threshold result recorded in the participant-bootstrap summary:
  - `phase2a_006`
  - threshold `0.51`
  - test balanced accuracy approximately `0.6098`

### 2. Updated outputs

- Updated:
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase2a_006_vs_phase3a_001_direct_comparison.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase2a_006_vs_phase3a_001_direct_comparison.md`

### 3. Corrected conclusion

- `phase3a_001` test balanced accuracy is approximately `0.6054`.
- `phase2a_006` official test balanced accuracy remains approximately `0.6098`.
- Therefore, `phase3a_001` remains close but does not replace `phase2a_006`.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("comparison csv:", REFERENCE_COMPARISON_CSV_PATH.relative_to(PROJECT_ROOT), REFERENCE_COMPARISON_CSV_PATH.exists())
print("comparison md:", REFERENCE_COMPARISON_MD_PATH.relative_to(PROJECT_ROOT), REFERENCE_COMPARISON_MD_PATH.exists())
print("\n=== Corrected Direct Comparison ===")
display(direct_comparison)
print("\nlog updated:", LOG_PATH, LOG_PATH.exists())
print("log appended characters:", len(log_entry))

comparison csv: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\phase2a_006_vs_phase3a_001_direct_comparison.csv True
comparison md: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\phase2a_006_vs_phase3a_001_direct_comparison.md True

=== Corrected Direct Comparison ===


,candidate_role,experiment_id,feature_timing,subset_name,model_family,window,selection_basis,selected_threshold,test_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,delta_balanced_accuracy_vs_phase2a_006
0,current_previous_day_reference,phase2a_006,previous_day,daytime_only,bilstm,14,validation-selected threshold recorded in boot...,0.51,0.609761,0.606261,0.578799,0.583333,0.589474,0.577320,0.000000
1,phase3a_close_rolling_trend_candidate,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,Phase 3A validation-selected threshold,0.35,0.605386,0.638470,0.525037,0.614458,0.488038,0.829268,-0.004375



log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
log appended characters: 1138


In [3]:
# Cell 11. Phase 3B seed robustness grid 생성
# 원하는 결과:
# - phase3a_001(GRU window 7), phase3a_002(GRU window 14)를 여러 seed로 반복할 준비를 한다.
# - 아직 training은 실행하지 않는다.

PHASE3B_OUTPUT_DIR = EXPERIMENT_DIR / "phase_3b_seed_robustness_outputs"
PHASE3B_MODEL_DIR = PHASE3B_OUTPUT_DIR / "models"
PHASE3B_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE3B_MODEL_DIR.mkdir(parents=True, exist_ok=True)

PHASE3B_METRICS_PATH = PHASE3B_OUTPUT_DIR / "phase_3b_seed_robustness_model_metrics.csv"
PHASE3B_HISTORY_PATH = PHASE3B_OUTPUT_DIR / "phase_3b_seed_robustness_training_history.csv"
PHASE3B_PREDICTIONS_PATH = PHASE3B_OUTPUT_DIR / "phase_3b_seed_robustness_model_predictions.csv"

PHASE3B_SEEDS = [7, 123, 2026]
PHASE3B_BASE_EXPERIMENTS = ["phase3a_001", "phase3a_002"]

phase3b_rows = []
for base_experiment_id in PHASE3B_BASE_EXPERIMENTS:
    base_config = loaded_experiments[base_experiment_id]["config"]
    for seed in PHASE3B_SEEDS:
        phase3b_rows.append(
            {
                "experiment_id": f"{base_experiment_id}_seed{seed}",
                "base_experiment_id": base_experiment_id,
                "seed": seed,
                "feature_timing": base_config["feature_timing"],
                "subset_name": base_config["subset_name"],
                "model_family": base_config["model_family"],
                "window": int(base_config["window"]),
                "tensor_dir": base_config["tensor_dir"],
                "feature_count": int(base_config["feature_count"]),
            }
        )

phase3b_grid = pd.DataFrame(phase3b_rows)

print("=== Phase 3B Seed Robustness Grid ===")
display(phase3b_grid)
print("output dir:", PHASE3B_OUTPUT_DIR.relative_to(PROJECT_ROOT))

=== Phase 3B Seed Robustness Grid ===


,experiment_id,base_experiment_id,seed,feature_timing,subset_name,model_family,window,tensor_dir,feature_count
0,phase3a_001_seed7,phase3a_001,7,previous_day,daytime_only_rolling_trend,gru,7,data\processed\deep_learning_previous_day\roll...,122
1,phase3a_001_seed123,phase3a_001,123,previous_day,daytime_only_rolling_trend,gru,7,data\processed\deep_learning_previous_day\roll...,122
2,phase3a_001_seed2026,phase3a_001,2026,previous_day,daytime_only_rolling_trend,gru,7,data\processed\deep_learning_previous_day\roll...,122
3,phase3a_002_seed7,phase3a_002,7,previous_day,daytime_only_rolling_trend,gru,14,data\processed\deep_learning_previous_day\roll...,122
4,phase3a_002_seed123,phase3a_002,123,previous_day,daytime_only_rolling_trend,gru,14,data\processed\deep_learning_previous_day\roll...,122
5,phase3a_002_seed2026,phase3a_002,2026,previous_day,daytime_only_rolling_trend,gru,14,data\processed\deep_learning_previous_day\roll...,122


output dir: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3b_seed_robustness_outputs


In [6]:
# Cell 12. Phase 3B seed robustness 학습 실행
# 원하는 결과:
# - phase3a_001, phase3a_002를 seed 7/123/2026으로 반복 학습한다.
# - validation balanced accuracy 기준 early stopping과 threshold selection을 유지한다.
# - 결과를 저장하고 seed별 test 성능 분포를 확인한다.

RUN_PHASE3B_SEED_ROBUSTNESS = True

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


if not RUN_PHASE3B_SEED_ROBUSTNESS:
    print("RUN_PHASE3B_SEED_ROBUSTNESS is False. Training was skipped.")
else:
    phase3b_all_metrics = []
    phase3b_all_history = []
    phase3b_all_predictions = []
    phase3b_saved_model_paths = []

    print("=== Phase 3B Seed Robustness Training Started ===")
    print("device:", DEVICE)
    print("runs:", len(phase3b_grid))

    for _, row in phase3b_grid.iterrows():
        run_experiment_id = row["experiment_id"]
        base_experiment_id = row["base_experiment_id"]
        seed = int(row["seed"])

        print("\n" + "=" * 80)
        print(f"Training {run_experiment_id} from {base_experiment_id} with seed={seed}")
        print("=" * 80)

        set_all_seeds(seed)

        result = train_one_experiment(
            experiment_id=run_experiment_id,
            bundle=loaded_experiments[base_experiment_id],
            device=DEVICE,
        )

        metrics = result["metrics"].copy()
        history = result["history"].copy()
        predictions = result["predictions"].copy()

        metrics["base_experiment_id"] = base_experiment_id
        metrics["seed"] = seed

        history["base_experiment_id"] = base_experiment_id
        history["seed"] = seed

        predictions["base_experiment_id"] = base_experiment_id
        predictions["seed"] = seed

        phase3b_all_metrics.append(metrics)
        phase3b_all_history.append(history)
        phase3b_all_predictions.append(predictions)
        phase3b_saved_model_paths.append(result["model_path"])

        # Phase 3B 전용 model dir에도 복사하지 않고, 기존 train_one_experiment 저장 위치를 기록합니다.
        print(f"saved model: {result['model_path'].relative_to(PROJECT_ROOT)}")

    phase3b_metrics = pd.concat(phase3b_all_metrics, ignore_index=True)
    phase3b_history = pd.concat(phase3b_all_history, ignore_index=True)
    phase3b_predictions = pd.concat(phase3b_all_predictions, ignore_index=True)

    phase3b_metrics.to_csv(PHASE3B_METRICS_PATH, index=False, encoding="utf-8-sig")
    phase3b_history.to_csv(PHASE3B_HISTORY_PATH, index=False, encoding="utf-8-sig")
    phase3b_predictions.to_csv(PHASE3B_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

    print("\n=== Saved Phase 3B Outputs ===")
    print("metrics:", PHASE3B_METRICS_PATH.relative_to(PROJECT_ROOT), PHASE3B_METRICS_PATH.exists())
    print("history:", PHASE3B_HISTORY_PATH.relative_to(PROJECT_ROOT), PHASE3B_HISTORY_PATH.exists())
    print("predictions:", PHASE3B_PREDICTIONS_PATH.relative_to(PROJECT_ROOT), PHASE3B_PREDICTIONS_PATH.exists())

    phase3b_test_metrics = (
        phase3b_metrics[phase3b_metrics["split"] == "test"]
        .sort_values(["base_experiment_id", "seed"])
        .reset_index(drop=True)
    )

    print("\n=== Phase 3B Test Metrics By Seed ===")
    display(
        phase3b_test_metrics[
            [
                "base_experiment_id",
                "experiment_id",
                "seed",
                "model_family",
                "window",
                "best_epoch",
                "selected_threshold_from_validation",
                "balanced_accuracy",
                "roc_auc",
                "average_precision",
                "f1",
                "precision",
                "recall",
                "tn",
                "fp",
                "fn",
                "tp",
            ]
        ]
    )

    phase3b_seed_summary = (
        phase3b_test_metrics
        .groupby(["base_experiment_id", "model_family", "window"])
        .agg(
            runs=("experiment_id", "count"),
            mean_test_balanced_accuracy=("balanced_accuracy", "mean"),
            std_test_balanced_accuracy=("balanced_accuracy", "std"),
            min_test_balanced_accuracy=("balanced_accuracy", "min"),
            max_test_balanced_accuracy=("balanced_accuracy", "max"),
            mean_test_roc_auc=("roc_auc", "mean"),
            mean_test_f1=("f1", "mean"),
            mean_test_precision=("precision", "mean"),
            mean_test_recall=("recall", "mean"),
        )
        .reset_index()
    )

    print("\n=== Phase 3B Seed Robustness Summary ===")
    display(phase3b_seed_summary)

    print("\nReference:")
    print("phase2a_006 previous-day reference test balanced accuracy = 0.6098")
    print("phase3a_001 original run test balanced accuracy = 0.6054")

=== Phase 3B Seed Robustness Training Started ===
device: cpu
runs: 6

Training phase3a_001_seed7 from phase3a_001 with seed=7
phase3a_001_seed7 epoch 001 | loss=0.8005 | val_bal_acc=0.5975 | val_auc=0.5919 | thr=0.52
phase3a_001_seed7 epoch 010 | loss=0.5982 | val_bal_acc=0.6354 | val_auc=0.6548 | thr=0.36
phase3a_001_seed7 epoch 020 | loss=0.4398 | val_bal_acc=0.5921 | val_auc=0.5760 | thr=0.46
phase3a_001_seed7 early stopped at epoch 24; best_epoch=4, best_val_bal_acc=0.6854
saved model: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_outputs\models\phase3a_001_seed7_gru_best.pt

Training phase3a_001_seed123 from phase3a_001 with seed=123
phase3a_001_seed123 epoch 001 | loss=0.8046 | val_bal_acc=0.6290 | val_auc=0.6246 | thr=0.47
phase3a_001_seed123 epoch 010 | loss=0.6100 | val_bal_acc=0.6320 | val_auc=0.6589 | thr=0.24
phase3a_001_seed123 epoch 020 | loss=0.4800 | val_bal_acc=0.6149 | val_auc=0.6221 | thr=0.45
phase3a_001_seed123 

,base_experiment_id,experiment_id,seed,model_family,window,best_epoch,selected_threshold_from_validation,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
0,phase3a_001,phase3a_001_seed7,7,gru,7,4,0.34,0.591569,0.666432,0.550187,0.610465,0.475113,0.853659,57,116,18,105
1,phase3a_001,phase3a_001_seed123,123,gru,7,3,0.32,0.624442,0.659476,0.551553,0.625387,0.505000,0.821138,74,99,22,101
2,phase3a_001,phase3a_001_seed2026,2026,gru,7,7,0.31,0.579820,0.626580,0.514781,0.586420,0.472637,0.772358,67,106,28,95
3,phase3a_002,phase3a_002_seed7,7,gru,14,8,0.32,0.505297,0.507519,0.500586,0.584980,0.474359,0.762887,27,82,23,74
4,phase3a_002,phase3a_002_seed123,123,gru,14,8,0.27,0.575239,0.591128,0.547174,0.628099,0.524138,0.783505,40,69,21,76
5,phase3a_002,phase3a_002_seed2026,2026,gru,14,8,0.37,0.486475,0.514424,0.505261,0.515837,0.459677,0.587629,42,67,40,57



=== Phase 3B Seed Robustness Summary ===


,base_experiment_id,model_family,window,runs,mean_test_balanced_accuracy,std_test_balanced_accuracy,min_test_balanced_accuracy,max_test_balanced_accuracy,mean_test_roc_auc,mean_test_f1,mean_test_precision,mean_test_recall
0,phase3a_001,gru,7,3,0.598611,0.023129,0.579820,0.624442,0.650829,0.607424,0.484250,0.815718
1,phase3a_002,gru,14,3,0.522337,0.046771,0.486475,0.575239,0.537690,0.576306,0.486058,0.711340



Reference:
phase2a_006 previous-day reference test balanced accuracy = 0.6098
phase3a_001 original run test balanced accuracy = 0.6054


In [7]:
# Cell 13. Phase 3B seed robustness summary 저장 및 log 업데이트
# 원하는 결과:
# - seed별 test metric과 base experiment별 robustness summary를 CSV/Markdown으로 저장한다.
# - phase2a_006 reference와 비교해 최종 해석을 기록한다.
# - log/2026-06-29.md를 업데이트한다.

PHASE3B_TEST_METRICS_PATH = PHASE3B_OUTPUT_DIR / "phase_3b_seed_robustness_test_metrics.csv"
PHASE3B_SEED_SUMMARY_PATH = PHASE3B_OUTPUT_DIR / "phase_3b_seed_robustness_summary.csv"
PHASE3B_SUMMARY_MD_PATH = PHASE3B_OUTPUT_DIR / "phase_3b_seed_robustness_summary.md"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

phase3b_metrics = pd.read_csv(PHASE3B_METRICS_PATH, encoding="utf-8-sig")

phase3b_test_metrics = (
    phase3b_metrics[phase3b_metrics["split"] == "test"]
    .sort_values(["base_experiment_id", "seed"])
    .reset_index(drop=True)
)

phase3b_seed_summary = (
    phase3b_test_metrics
    .groupby(["base_experiment_id", "model_family", "window"])
    .agg(
        runs=("experiment_id", "count"),
        mean_test_balanced_accuracy=("balanced_accuracy", "mean"),
        std_test_balanced_accuracy=("balanced_accuracy", "std"),
        min_test_balanced_accuracy=("balanced_accuracy", "min"),
        max_test_balanced_accuracy=("balanced_accuracy", "max"),
        mean_test_roc_auc=("roc_auc", "mean"),
        mean_test_f1=("f1", "mean"),
        mean_test_precision=("precision", "mean"),
        mean_test_recall=("recall", "mean"),
    )
    .reset_index()
)

REFERENCE_BAL_ACC = 0.609761

phase3b_seed_summary["delta_mean_bal_acc_vs_phase2a_006"] = (
    phase3b_seed_summary["mean_test_balanced_accuracy"] - REFERENCE_BAL_ACC
)
phase3b_seed_summary["beats_phase2a_006_by_mean"] = (
    phase3b_seed_summary["mean_test_balanced_accuracy"] > REFERENCE_BAL_ACC
)

phase3b_test_metrics["delta_bal_acc_vs_phase2a_006"] = (
    phase3b_test_metrics["balanced_accuracy"] - REFERENCE_BAL_ACC
)
phase3b_test_metrics["beats_phase2a_006"] = (
    phase3b_test_metrics["balanced_accuracy"] > REFERENCE_BAL_ACC
)

phase3b_test_metrics.to_csv(PHASE3B_TEST_METRICS_PATH, index=False, encoding="utf-8-sig")
phase3b_seed_summary.to_csv(PHASE3B_SEED_SUMMARY_PATH, index=False, encoding="utf-8-sig")

summary_md = """# Phase 3B Seed Robustness Summary

## Purpose

Phase 3B repeated the closest Phase 3A GRU candidates across additional random seeds to check whether the rolling/trend result was stable.

## Reference

The current strict previous-day reference remains:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

Reference test balanced accuracy: `0.6098`

## Seed Robustness Summary

| base_experiment_id | model | window | runs | mean test balanced accuracy | std | min | max | mean ROC AUC | mean F1 | mean precision | mean recall | delta mean vs phase2a_006 |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
"""

for _, row in phase3b_seed_summary.iterrows():
    summary_md += (
        f"| `{row['base_experiment_id']}` | `{row['model_family']}` | {int(row['window'])} | "
        f"{int(row['runs'])} | {float(row['mean_test_balanced_accuracy']):.4f} | "
        f"{float(row['std_test_balanced_accuracy']):.4f} | "
        f"{float(row['min_test_balanced_accuracy']):.4f} | "
        f"{float(row['max_test_balanced_accuracy']):.4f} | "
        f"{float(row['mean_test_roc_auc']):.4f} | "
        f"{float(row['mean_test_f1']):.4f} | "
        f"{float(row['mean_test_precision']):.4f} | "
        f"{float(row['mean_test_recall']):.4f} | "
        f"{float(row['delta_mean_bal_acc_vs_phase2a_006']):.4f} |\n"
    )

summary_md += """

## Interpretation

`phase3a_001` remained the stronger Phase 3A rolling/trend candidate, but its mean seed-robust test balanced accuracy did not exceed `phase2a_006`.

`phase3a_002` was weaker and less stable across seeds, supporting the interpretation that its high validation score in the original Phase 3A run did not transfer reliably.

## Conclusion

Phase 3B does not justify replacing the strict previous-day reference candidate.

Current strict previous-day reference:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

Close but not replacement candidate:

`phase3a_001 / previous_day / daytime_only_rolling_trend / window 7 / GRU`
"""

PHASE3B_SUMMARY_MD_PATH.write_text(summary_md, encoding="utf-8")

log_entry = """

## Phase 3B Seed Robustness Check

### 1. Reason for update

- Ran a seed robustness check for the closest Phase 3A rolling/trend GRU candidates.
- Repeated:
  - `phase3a_001`: GRU window 7
  - `phase3a_002`: GRU window 14
- Seeds:
  - `7`
  - `123`
  - `2026`

### 2. Created outputs

- Created:
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_model_metrics.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_training_history.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_model_predictions.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_test_metrics.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_summary.csv`
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_summary.md`

### 3. Main results

- `phase3a_001` seed robustness:
  - mean test balanced accuracy approximately `0.5986`
  - range approximately `[0.5798, 0.6244]`
  - mean remains below `phase2a_006` reference test balanced accuracy `0.6098`
- `phase3a_002` seed robustness:
  - mean test balanced accuracy approximately `0.5223`
  - weaker and less stable than `phase3a_001`

### 4. Current conclusion

- Phase 3B does not justify replacing the strict previous-day reference candidate.
- `phase2a_006` remains the current strict previous-day reference.
- `phase3a_001` remains a close rolling/trend comparison candidate.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("test metrics:", PHASE3B_TEST_METRICS_PATH.relative_to(PROJECT_ROOT), PHASE3B_TEST_METRICS_PATH.exists())
print("seed summary:", PHASE3B_SEED_SUMMARY_PATH.relative_to(PROJECT_ROOT), PHASE3B_SEED_SUMMARY_PATH.exists())
print("summary md:", PHASE3B_SUMMARY_MD_PATH.relative_to(PROJECT_ROOT), PHASE3B_SUMMARY_MD_PATH.exists())

print("\n=== Phase 3B Seed Summary Saved ===")
display(phase3b_seed_summary)

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())
print("log appended characters:", len(log_entry))

test metrics: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3b_seed_robustness_outputs\phase_3b_seed_robustness_test_metrics.csv True
seed summary: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3b_seed_robustness_outputs\phase_3b_seed_robustness_summary.csv True
summary md: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3b_seed_robustness_outputs\phase_3b_seed_robustness_summary.md True

=== Phase 3B Seed Summary Saved ===


,base_experiment_id,model_family,window,runs,mean_test_balanced_accuracy,std_test_balanced_accuracy,min_test_balanced_accuracy,max_test_balanced_accuracy,mean_test_roc_auc,mean_test_f1,mean_test_precision,mean_test_recall,delta_mean_bal_acc_vs_phase2a_006,beats_phase2a_006_by_mean
0,phase3a_001,gru,7,3,0.598611,0.023129,0.579820,0.624442,0.650829,0.607424,0.484250,0.815718,-0.011150,False
1,phase3a_002,gru,14,3,0.522337,0.046771,0.486475,0.575239,0.537690,0.576306,0.486058,0.711340,-0.087424,False



log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
log appended characters: 1936
